# Deploying Custom Models on Databricks Model Serving
In the case that you are utilizing a framework that is not natively part of the MLflow logging capabilities, you can utilize MLflow's PyFunc model flavor to deploy custom models, frameworks, and code. There are a few method signatures you must adhere to, but you can utilize this to bring your own (BYO) custom models and frameworks essentially. 

Other examples of this might be leveraging this with something like vLLM to customize your LLM model serving. In this example we showcase how you can bring a 3P Python framework and still leverage model serving endpoints.
### Additional Resources
- MLflow Pyfunc: https://mlflow.org/docs/latest/api_reference/python_api/mlflow.pyfunc.html
- Custom Models Databricks: https://docs.databricks.com/aws/en/machine-learning/model-serving/custom-models

## Setup

In [0]:
%pip install -U mlflow databricks-sdk spacy --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
databricks-feature-engineering 0.12.1 requires protobuf<6,>=3.12.0, but you have protobuf 6.33.5 which is incompatible.
google-api-core 2.20.0 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0.dev0,>=3.19.5, but you have protobuf 6.33.5 which is incompatible.
googleapis-common-protos 1.65.0 requires protobuf!=3.20.0,!=3.20.1,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0.dev0,>=3.20.2, but you have protobuf 6.33.5 which is incompatible.
grpcio-status 1.67.0 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.5 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.5 which is incompatible.
Note: you may need to restart the kernel using %restart_python o

In [0]:
dbutils.library.restartPython()

In [0]:
# Unity Catalog destination
CATALOG = "main"                # <-- change

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS main.langid_custom;

## Pyfunc Setup

In [0]:
import json
import mlflow
import mlflow.pyfunc

class LangIdPyFunc(mlflow.pyfunc.PythonModel):
    """
    Minimal MLflow PyFunc that detects language using 'langid'.
    Input: DataFrame or dict with 'text'
    Output: JSON string, e.g., {"lang": "en", "score": 0.98}
    """
    def load_context(self, context):
        import langid
        self.langid = langid

    def predict(self, context, model_input):
        # Accept pandas DataFrame or dict with 'text'
        if hasattr(model_input, "iloc"):
            text = model_input["text"].iloc[0]
        elif isinstance(model_input, dict):
            text = model_input.get("text")
        else:
            raise ValueError("Expected DataFrame column 'text' or dict key 'text'.")
        lang, score = self.langid.classify(text)
        return json.dumps({"lang": lang, "score": float(score)})

/local_disk0/.ephemeral_nfs/envs/pythonEnv-ec30829f-7040-40f3-a9c8-9b7871626bd1/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


## Log & Register Model

In [0]:
from mlflow.models.signature import ModelSignature, Schema, ColSpec
import mlflow

signature = ModelSignature(
    inputs=Schema([ColSpec("string", "text")]),
    outputs=Schema([ColSpec("string", "lang_json")])
)

with mlflow.start_run():
    info = mlflow.pyfunc.log_model(
        name="langid_pyfunc",                 # MLflow 3-style name
        python_model=LangIdPyFunc(),
        signature=signature,
        pip_requirements=[
            "mlflow>=2.4.0",
            "langid"                          # small pure-Python lib
        ],
        input_example={"text": "OpenAI is based in San Francisco."}
    )
print("model_uri:", info.model_uri)

🔗 View Logged Model at: https://e2-demo-field-eng.cloud.databricks.com/ml/experiments/25844094986597/models/m-7d63707043cc4126b39d1625d0bb791e?o=1444828305810485
2026/02/13 17:49:01 INFO mlflow.pyfunc: Validating input example against model signature
2026/02/13 17:49:01 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - langid (current: uninstalled, required: langid)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.
2026/02/13 17:49:01 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - langid (current: uninstalled, required: langid)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the res

model_uri: models:/m-7d63707043cc4126b39d1625d0bb791e


In [0]:
CATALOG, SCHEMA, NAME = "main", "langid_custom", "langid_pyfunc"
UC_FULL_NAME = f"{CATALOG}.{SCHEMA}.{NAME}"
registered = mlflow.register_model(info.model_uri, UC_FULL_NAME)
print(f"Registered: {UC_FULL_NAME} v{registered.version}")

Successfully registered model 'main.langid_custom.langid_pyfunc'.


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

🔗 Created version '1' of model 'main.langid_custom.langid_pyfunc': https://e2-demo-field-eng.cloud.databricks.com/explore/data/models/main/langid_custom/langid_pyfunc/version/1?o=1444828305810485


Registered: main.langid_custom.langid_pyfunc v1


## Deploy Model

In [0]:
from mlflow.deployments import get_deploy_client
client = get_deploy_client("databricks")

endpoint_name = "langid-endpoint"
client.create_endpoint(
    name=endpoint_name,
    config={
        "served_entities": [{
            "name": NAME,
            "entity_name": UC_FULL_NAME,
            "entity_version": str(registered.version),
            "workload_type": "CPU",
            "workload_size": "Small",
            "scale_to_zero_enabled": True
        }]
    }
)
print("Endpoint creation requested.")

/local_disk0/.ephemeral_nfs/envs/pythonEnv-ec30829f-7040-40f3-a9c8-9b7871626bd1/lib/python3.12/site-packages/mlflow/deployments/databricks/__init__.py:489: UserWarning: Passing 'name', 'config', and 'route_optimized' as separate parameters is deprecated. Please pass the full API request payload as a single dictionary in the 'config' parameter.
  warnings.warn("\n".join(warnings_list), UserWarning)


Endpoint creation requested.


In [0]:
import time
def poll_ready(name, timeout_s=900, poll_s=45):
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        ep = client.get_endpoint(name)
        ready = ep.get("state", {}).get("ready")
        print(f"ready={ready}")
        if ready == "READY": return ep
        if ready == "FAILED": raise RuntimeError(f"Endpoint failed:\n{ep}")
        time.sleep(poll_s)
    raise TimeoutError("Timeout waiting for READY.")
_ = poll_ready(endpoint_name)

ready=NOT_READY
ready=NOT_READY
ready=NOT_READY
ready=NOT_READY
ready=NOT_READY
ready=NOT_READY
ready=NOT_READY
ready=NOT_READY
ready=NOT_READY
ready=READY


## Sample Inference

In [0]:
payload = {"dataframe_records": [{"text": "Who is Roger Federer"}]}
result = client.predict(endpoint=endpoint_name, inputs=payload)
print(result)

{'predictions': '{"lang": "en", "score": -63.18179702758789}'}


In [0]:
%%time
for i in range(100):
  payload = {"dataframe_records": [{"text": "Who is Roger Federer"}]}
  result = client.predict(endpoint=endpoint_name, inputs=payload)

CPU times: user 266 ms, sys: 136 ms, total: 402 ms
Wall time: 2.71 s
